# TODO:
- Prep data
  - get ob data
  - get OHLCV data for ob time
  - Quantise the OB and Candle data
- import that into a pandas df
- start completing theOne's functions

In [651]:
root_path = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.append(root_path)
from theOne import theOne
import pandas as pd


# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal

import pandas as pd
from core.data_sources.clob import CLOBDataSource
from core.data_structures.candles import Candles

In [652]:
root_path

'/Users/schalkvisagie/csHonours/project/25349589-MN8-src/quants-lab'

## Load the Candles OHLCV

In [653]:
# Load Candles
clob = CLOBDataSource()
CONNECTOR_NAME = "binance"
INTERVALS = "1s"
trading_pair = "POL-USDT"
# DAYS = 60

clob.load_candles_cache(root_path)
all_candles = clob.get_candles_from_cache(CONNECTOR_NAME, trading_pair, INTERVALS)
print(all_candles)

candles: Candles = all_candles
candlesdf = candles.data

# candlesdf
# filtered_df = df[df['quote_asset_volume'] > 0]
# print(filtered_df)

2025-05-31 17:32:11,648 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x32e112bd0>


In [654]:
def load_market_data(connector_name: str, trading_pair: str, data_type: str = "order_book") -> pd.DataFrame:
    """
    Load market data from files for a specific connector and trading pair.
    
    Args:
        connector_name: Name of the connector (e.g., "bitmart_paper_trade")
        trading_pair: Trading pair symbol (e.g., "LINK-USDT")
        data_type: Type of data to load ("order_book" or "trades")
    
    Returns:
        pd.DataFrame: Concatenated DataFrame containing all data from matching files
    """
    folder = root_path + f"/data/order_book/"
    
    # Define the pattern based on data type
    pattern = "order_book_snapshots" if data_type == "order_book" else "trades"
    
    # Find all matching files
    files = [
        file for file in os.listdir(folder) 
        if connector_name in file 
        and trading_pair in file 
        and pattern in file
    ]
    
    if not files:
        raise FileNotFoundError(f"No {data_type} files found for {connector_name} {trading_pair}")
    
    # Load and concatenate all matching files
    dfs = []
    for file in files:
        df = pd.read_json(folder + "/" + file, lines=True)
        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)

# Example usage:
order_book_df = load_market_data(CONNECTOR_NAME, trading_pair, "order_book")
# trades_df = load_market_data("binance", "POL-USDT", "trades")


In [655]:
order_book_df.rename(columns={"ts": "timestamp"}, inplace=True)
order_book_df

,timestamp,bids,asks
0,1748699400,"[[0.2111, 8741.3], [0.211, 37744.6], [0.2109, ...","[[0.2112, 12250.1], [0.21130000000000002, 4998..."
1,1748699401,"[[0.2111, 9688.5], [0.211, 37744.6], [0.2109, ...","[[0.2112, 14207.3], [0.21130000000000002, 5420..."
2,1748699402,"[[0.2111, 18758.0], [0.211, 37744.6], [0.2109,...","[[0.2112, 9235.1], [0.21130000000000002, 51279..."
3,1748699403,"[[0.2111, 18758.0], [0.211, 37744.6], [0.2109,...","[[0.2112, 9235.1], [0.21130000000000002, 51279..."
4,1748699404,"[[0.2111, 18770.1], [0.211, 37744.6], [0.2109,...","[[0.2112, 9008.9], [0.21130000000000002, 42093..."
...,...,...,...
6126,1748705523,"[[0.2124, 3523.9], [0.21230000000000002, 49484...","[[0.21250000000000002, 20525.6], [0.2126, 4713..."
6127,1748705524,"[[0.21230000000000002, 25389.7], [0.2122, 3928...","[[0.2124, 7866.1], [0.21250000000000002, 38705..."
6128,1748705525,"[[0.21230000000000002, 25389.7], [0.2122, 5341...","[[0.2124, 7866.1], [0.21250000000000002, 35626..."
6129,1748705526,"[[0.21230000000000002, 25389.7], [0.2122, 5341...","[[0.2124, 7510.6], [0.21250000000000002, 35626..."


### Quantise the orderbook data (Fit to each second)

In [656]:
candlesdf
order_book_df

# Ensure timestamp columns are of the same type (int)
candlesdf['timestamp'] = candlesdf['timestamp'].astype(int)
order_book_df['timestamp'] = order_book_df['timestamp'].astype(int)

# Merge on 'timestamp'
merged_df = pd.merge(
    candlesdf,
    order_book_df,
    on='timestamp',
    how='inner',  # Only keep rows with matching timestamps
    suffixes=('_candle', '_orderbook')
)

# Display the merged DataFrame
merged_df['datetime'] = pd.to_datetime(merged_df['timestamp'], unit='s')
# merged_df.head()
merged_df

,timestamp,open,high,low,close,volume,quote_asset_volume,n_trades,taker_buy_base_volume,taker_buy_quote_volume,bids,asks,datetime
0,1748699400,0.2112,0.2112,0.2112,0.2112,0,0,0,0,0,"[[0.2111, 8741.3], [0.211, 37744.6], [0.2109, ...","[[0.2112, 12250.1], [0.21130000000000002, 4998...",2025-05-31 13:50:00
1,1748699401,0.2112,0.2112,0.2112,0.2112,0,0,0,0,0,"[[0.2111, 9688.5], [0.211, 37744.6], [0.2109, ...","[[0.2112, 14207.3], [0.21130000000000002, 5420...",2025-05-31 13:50:01
2,1748699402,0.2112,0.2112,0.2112,0.2112,0,0,0,0,0,"[[0.2111, 18758.0], [0.211, 37744.6], [0.2109,...","[[0.2112, 9235.1], [0.21130000000000002, 51279...",2025-05-31 13:50:02
3,1748699403,0.2112,0.2112,0.2112,0.2112,0,0,0,0,0,"[[0.2111, 18758.0], [0.211, 37744.6], [0.2109,...","[[0.2112, 9235.1], [0.21130000000000002, 51279...",2025-05-31 13:50:03
4,1748699404,0.2112,0.2112,0.2112,0.2112,0,0,0,0,0,"[[0.2111, 18770.1], [0.211, 37744.6], [0.2109,...","[[0.2112, 9008.9], [0.21130000000000002, 42093...",2025-05-31 13:50:04
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6022,1748705419,0.2123,0.2123,0.2123,0.2123,0,0,0,0,0,"[[0.2122, 19598.7], [0.2121, 65739.1], [0.212,...","[[0.21230000000000002, 195.7], [0.2124, 34352....",2025-05-31 15:30:19
6023,1748705420,0.2123,0.2123,0.2123,0.2123,148.5,31.52655,3,148.5,31.52655,"[[0.2122, 19598.7], [0.2121, 70959.2], [0.212,...","[[0.21230000000000002, 195.7], [0.2124, 34352....",2025-05-31 15:30:20
6024,1748705421,0.2123,0.2123,0.2123,0.2123,0,0,0,0,0,"[[0.21230000000000002, 6048.4], [0.2122, 39641...","[[0.2124, 9538.5], [0.21250000000000002, 43763...",2025-05-31 15:30:21
6025,1748705422,0.2123,0.2123,0.2123,0.2123,0,0,0,0,0,"[[0.21230000000000002, 3015.0], [0.2122, 41131...","[[0.2124, 26275.9], [0.21250000000000002, 4376...",2025-05-31 15:30:22


Look for missing seconds

In [657]:
# Get the range of timestamps
min_ts = merged_df['timestamp'].min()
max_ts = merged_df['timestamp'].max()

# Create a set of all expected timestamps
expected_timestamps = set(range(min_ts, max_ts + 1))
actual_timestamps = set(merged_df['timestamp'])

# Find missing timestamps
missing_timestamps = expected_timestamps - actual_timestamps

print(f"Number of missing seconds: {len(missing_timestamps)}")
if missing_timestamps:
    print(f"Example missing timestamps: {sorted(list(missing_timestamps))[:10]}")
else:
    print("No missing seconds. Every second has an entry.")

Number of missing seconds: 0
No missing seconds. Every second has an entry.


## Add additional collumns

In [658]:
merged_df['taker_sell_base_volume'] = merged_df['volume'] - merged_df['taker_buy_base_volume']

merged_df

,timestamp,open,high,low,close,volume,quote_asset_volume,n_trades,taker_buy_base_volume,taker_buy_quote_volume,bids,asks,datetime,taker_sell_base_volume
0,1748699400,0.2112,0.2112,0.2112,0.2112,0,0,0,0,0,"[[0.2111, 8741.3], [0.211, 37744.6], [0.2109, ...","[[0.2112, 12250.1], [0.21130000000000002, 4998...",2025-05-31 13:50:00,0
1,1748699401,0.2112,0.2112,0.2112,0.2112,0,0,0,0,0,"[[0.2111, 9688.5], [0.211, 37744.6], [0.2109, ...","[[0.2112, 14207.3], [0.21130000000000002, 5420...",2025-05-31 13:50:01,0
2,1748699402,0.2112,0.2112,0.2112,0.2112,0,0,0,0,0,"[[0.2111, 18758.0], [0.211, 37744.6], [0.2109,...","[[0.2112, 9235.1], [0.21130000000000002, 51279...",2025-05-31 13:50:02,0
3,1748699403,0.2112,0.2112,0.2112,0.2112,0,0,0,0,0,"[[0.2111, 18758.0], [0.211, 37744.6], [0.2109,...","[[0.2112, 9235.1], [0.21130000000000002, 51279...",2025-05-31 13:50:03,0
4,1748699404,0.2112,0.2112,0.2112,0.2112,0,0,0,0,0,"[[0.2111, 18770.1], [0.211, 37744.6], [0.2109,...","[[0.2112, 9008.9], [0.21130000000000002, 42093...",2025-05-31 13:50:04,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6022,1748705419,0.2123,0.2123,0.2123,0.2123,0,0,0,0,0,"[[0.2122, 19598.7], [0.2121, 65739.1], [0.212,...","[[0.21230000000000002, 195.7], [0.2124, 34352....",2025-05-31 15:30:19,0
6023,1748705420,0.2123,0.2123,0.2123,0.2123,148.5,31.52655,3,148.5,31.52655,"[[0.2122, 19598.7], [0.2121, 70959.2], [0.212,...","[[0.21230000000000002, 195.7], [0.2124, 34352....",2025-05-31 15:30:20,0
6024,1748705421,0.2123,0.2123,0.2123,0.2123,0,0,0,0,0,"[[0.21230000000000002, 6048.4], [0.2122, 39641...","[[0.2124, 9538.5], [0.21250000000000002, 43763...",2025-05-31 15:30:21,0
6025,1748705422,0.2123,0.2123,0.2123,0.2123,0,0,0,0,0,"[[0.21230000000000002, 3015.0], [0.2122, 41131...","[[0.2124, 26275.9], [0.21250000000000002, 4376...",2025-05-31 15:30:22,0


In [659]:

def calculate_fills(merged_df, my_bids=None, my_asks=None):
    """
    Calculate order fills for each row in merged_df for given buy/sell orders.
    Returns a list of dicts with fill info for each order at each timestamp.
    """
    fills = []

    for idx in range(len(merged_df) - 1):  # -1 because we look ahead 1 second
        row = merged_df.iloc[idx]
        next_row = merged_df.iloc[idx + 1]

        # BUY LIMIT ORDERS
        if my_bids:
            for price, qty in my_bids:
                # Order book at T₀
                bids = row['bids']  # list of [price, volume]
                # Calculate V_above and V_same
                V_above = sum(v for p, v in bids if p > price)
                V_same = sum(v for p, v in bids if p == price)
                # S = taker_sell_base_volume at T₁
                S = next_row['taker_sell_base_volume']
                filled = max(0, min(qty, S - V_above - V_same))
                fills.append({
                    'timestamp': row['timestamp'],
                    'side': 'buy',
                    'price': price,
                    'qty': qty,
                    'filled': max(0, filled)
                })

        # SELL LIMIT ORDERS
        if my_asks:
            for price, qty in my_asks:
                asks = row['asks']
                V_below = sum(v for p, v in asks if p < price)
                V_same = sum(v for p, v in asks if p == price)
                B = next_row['taker_buy_base_volume']
                filled = max(0, min(qty, B - V_below - V_same))
                fills.append({
                    'timestamp': row['timestamp'],
                    'side': 'sell',
                    'price': price,
                    'qty': qty,
                    'filled': max(0, filled)
                })

    return fills

# Example usage:
my_bids = [[0.2112, 500]]
my_asks = [[0.2111, 200]]
fills = calculate_fills(merged_df, my_bids=my_bids, my_asks=my_asks)
fills_df = pd.DataFrame(fills)
fills_df

,timestamp,side,price,qty,filled
0,1748699400,buy,0.2112,500,0
1,1748699400,sell,0.2111,200,0
2,1748699401,buy,0.2112,500,0
3,1748699401,sell,0.2111,200,0
4,1748699402,buy,0.2112,500,0
...,...,...,...,...,...
12047,1748705420,sell,0.2111,200,0
12048,1748705421,buy,0.2112,500,0
12049,1748705421,sell,0.2111,200,0
12050,1748705422,buy,0.2112,500,0


In [660]:
exchanges = ["binance"]
trading_pair = "POL-USDT"

# get all the order book files and trades files
# unpack the best bid and best ask for the columns bids and asks of the dataframe (is the first observation of the list that is a tuple (amount, price))
# plot with scatter the best bid and best ask over time
# plot with scatter the trades over time
# plot with scatter the mid price over time
# plot with scatter the bid ask spread over time

ob_data = {}
trades_data = {}

for exchange in exchanges:
    # Load the order book data
    ob = load_market_data(exchange, trading_pair, "order_book")
    ob["best_bid_price"] = ob["bids"].apply(lambda x: x[0][0])
    ob["best_ask_price"] = ob["asks"].apply(lambda x: x[0][0])
    ob["best_bid_amount"] = ob["bids"].apply(lambda x: x[0][1])
    ob["best_ask_amount"] = ob["asks"].apply(lambda x: x[0][1])
    ob.index = pd.to_datetime(ob["ts"], unit="s")
    ob_data[exchange] = ob
    # Load the trades data
    trades = load_market_data(exchange, trading_pair, "trades")
    trades["amount_quote"] = trades["q_base"] * trades["price"]
    trades.index = pd.to_datetime(trades["ts"], unit="s")
    trades_data[exchange] = trades

In [661]:
# trades_data["binance"]


In [662]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_market_data(ob_df: pd.DataFrame, trades_df: pd.DataFrame, exchange: str, levels_to_plot: int = 10):
    """
    Create a plotly figure with order book and trades data.
    """
    fig = go.Figure()
    for level in range(1, levels_to_plot + 1):
        bids_price = ob_df["bids"].apply(lambda x: x[level][0])
        bids_amount = ob_df["bids"].apply(lambda x: x[level][1])
        asks_price = ob_df["asks"].apply(lambda x: x[level][0])
        asks_amount = ob_df["asks"].apply(lambda x: x[level][1])
        
        # Plot the bid line and add with color intensity based on amount

        fig.add_trace(
            go.Scatter(
                x=ob_df.index,
                y=bids_price,
                mode='lines',
                name=f'Bid {level}',
                line=dict(width=1, color='green'),
                marker=dict(
                    size=6,
                    color=bids_amount,
                    colorscale='Greens',  # Changed to Greens for consistency
                    showscale=True,
                    colorbar=dict(title='Bid Amount')
                )
            )
        )
        fig.add_trace(
            go.Scatter(
                x=ob_df.index,
                y=asks_price,
                mode='lines',
                name=f'Ask {level}',
                line=dict(width=1, color='red'),
                marker=dict(
                    size=6,
                    color=asks_amount,
                    colorscale='Reds',  # Kept as Reds for consistency
                    showscale=True,
                    colorbar=dict(title='Ask Amount')
                )
            )
        )

    
    # Trades trace remains unchanged
    fig.add_trace(
        go.Scatter(
            x=trades_df.index,
            y=trades_df['price'],
            mode='markers',
            name='Trades',
            marker=dict(
                # size=trades_df['amount_quote'] * 2,
                symbol='circle',
                color='blue',
                opacity=0.5
            )
        )
    )
    
    # Update layout
    fig.update_layout(
        title=f'Market Data for {exchange}',
        xaxis_title='Timestamp',
        yaxis_title='Price',
        showlegend=True
    )
    
    return fig
# Create plots for each exchange
for exchange in exchanges:
    fig = plot_market_data(
        ob_df=ob_data[exchange],
        trades_df=trades_data[exchange],
        exchange=exchange,
        levels_to_plot=1
    )
    fig.show()

# Backtest:

#### Backtest setup

In [663]:
def compute_volume_before(row, price, side='buy'):
    book = row['bids'] if side == 'buy' else row['asks']
    if not book:
        return 0.0

    if side == 'buy':
        return sum(qty for p, qty in book if p >= price)
    else:  # sell
        return sum(qty for p, qty in book if p <= price)


# Define your order
my_price = 0.21181
my_side = 'sell'  # or 'sell'

# Compute Volume_before for every row
merged_df['Volume_before'] = merged_df.apply(
    lambda row: compute_volume_before(row, price=my_price, side=my_side),
    axis=1
)
# merged_df



Calculating Order fills for each level provided


In [664]:
my_bids = [[0.1, 25], [0.09, 35], ... ]